# A06 - Deep Feelings

## 1. Setup

In [32]:
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

import nltk
for _corpus in ['stopwords', 'punkt', 'punkt_tab', 'wordnet', 'averaged_perceptron_tagger_eng']:
    nltk.download(_corpus, quiet=True)

from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

import spacy

RANDOM_STATE = 2026

## 2. Load data

Both CSVs are headerless: `label,text` with labels in `{-1, 0, 1}`.

In [33]:
# Assisted by Claude.

def load_split(path):
    df = pd.read_csv(path, header=None, names=['label', 'text'])
    # Fail gracefully if anything malformed slips through.
    df = df.dropna(subset=['text']).copy()
    df['text'] = df['text'].astype(str)
    return df
    
train_df = load_split('A06_train.csv')
test_df  = load_split('A06_test.csv')

X_train_text, y_train = train_df['text'].values, train_df['label'].values
X_test_text,  y_test  = test_df['text'].values,  test_df['label'].values

print(f'Train: {len(train_df):,}   Test: {len(test_df):,}')
print('Train label distribution:')
print(train_df['label'].value_counts().sort_index())

Train: 26,499   Test: 6,625
Train label distribution:
label
-1     5368
 0    11510
 1     9621
Name: count, dtype: int64


## 3. Logistic Regression Classifier

All three systems use the same logistic regression classifier so that performance differences come from the features, not the model.

In [34]:
# Assisted by Claude.

def make_clf():
    # max_iter bumped so the solver converges on the larger BOW feature spaces.
    return LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)

def evaluate(name, X_tr, X_te, y_tr, y_te):
    clf = make_clf()
    clf.fit(X_tr, y_tr)
    preds = clf.predict(X_te)
    acc = accuracy_score(y_te, preds)
    print(f'\n=== {name} ===')
    print(f'Test accuracy: {acc:.4f}')
    print(classification_report(y_te, preds, digits=4))
    return acc

results = {}

## 4. Baseline System

`CountVectorizer` with default settings.

In [35]:
# Assisted by Claude.

baseline_vec = CountVectorizer()
X_tr_bow = baseline_vec.fit_transform(X_train_text)
X_te_bow = baseline_vec.transform(X_test_text)
print(f'BOW vocab size: {len(baseline_vec.vocabulary_):,}')
results['baseline'] = evaluate('Baseline (BOW)', X_tr_bow, X_te_bow, y_train, y_test)

BOW vocab size: 53,586

=== Baseline (BOW) ===
Test accuracy: 0.6572
              precision    recall  f1-score   support

          -1     0.6342    0.5337    0.5797      1319
           0     0.6266    0.7035    0.6628      2877
           1     0.7116    0.6694    0.6899      2429

    accuracy                         0.6572      6625
   macro avg     0.6575    0.6356    0.6441      6625
weighted avg     0.6593    0.6572    0.6562      6625



## 5. Enhanced System

Adds two preprocessing steps to the baseline BOW: NLTK stopword removal and POS-aware WordNet lemmatization. Each is evaluated alone and together so we can see whether the combination actually helps.

In [36]:
# Assisted by Claude.

STOP_WORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# NLTK uses Penn-Treebank tags; WordNet uses its own short tags. Map between them.
_POS_MAP = {'J': wordnet.ADJ, 'V': wordnet.VERB, 'N': wordnet.NOUN, 'R': wordnet.ADV}
def _wn_pos(tag):
    return _POS_MAP.get(tag[0], wordnet.NOUN)

_WORD_RE = re.compile(r"[A-Za-z']+")

def preprocess(text, remove_stop=False, lemmatize=False):
    """Tokenize a single document and optionally drop stopwords / lemmatize.
    Returns a space-joined string so it can feed straight into CountVectorizer.
    """
    tokens = [t.lower() for t in _WORD_RE.findall(text)]
    if lemmatize:
        # POS-tag once for the whole sentence so verbs/adjectives lemmatize correctly.
        tokens = [lemmatizer.lemmatize(tok, _wn_pos(tag))
                  for tok, tag in pos_tag(tokens)]
    if remove_stop:
        tokens = [t for t in tokens if t not in STOP_WORDS]
    return ' '.join(tokens)

def build_bow(train_texts, test_texts, remove_stop, lemmatize):
    tr = [preprocess(t, remove_stop, lemmatize) for t in train_texts]
    te = [preprocess(t, remove_stop, lemmatize) for t in test_texts]
    vec = CountVectorizer()
    return vec.fit_transform(tr), vec.transform(te), vec

In [37]:
# Stopwords only
Xtr, Xte, vec = build_bow(X_train_text, X_test_text, remove_stop=True, lemmatize=False)
print(f'Vocab size (stopwords removed): {len(vec.vocabulary_):,}')
results['enhanced_stopwords'] = evaluate('Enhanced: stopwords removed', Xtr, Xte, y_train, y_test)

Vocab size (stopwords removed): 52,542

=== Enhanced: stopwords removed ===
Test accuracy: 0.6465
              precision    recall  f1-score   support

          -1     0.6131    0.4890    0.5441      1319
           0     0.6175    0.7021    0.6571      2877
           1     0.7029    0.6661    0.6840      2429

    accuracy                         0.6465      6625
   macro avg     0.6445    0.6191    0.6284      6625
weighted avg     0.6479    0.6465    0.6445      6625



In [38]:
# Lemmatization only
Xtr, Xte, vec = build_bow(X_train_text, X_test_text, remove_stop=False, lemmatize=True)
print(f'Vocab size (lemmatized): {len(vec.vocabulary_):,}')
results['enhanced_lemma'] = evaluate('Enhanced: WordNet lemmatized', Xtr, Xte, y_train, y_test)

Vocab size (lemmatized): 48,504

=== Enhanced: WordNet lemmatized ===
Test accuracy: 0.6610
              precision    recall  f1-score   support

          -1     0.6328    0.5292    0.5764      1319
           0     0.6345    0.7073    0.6690      2877
           1     0.7110    0.6776    0.6939      2429

    accuracy                         0.6610      6625
   macro avg     0.6595    0.6381    0.6464      6625
weighted avg     0.6622    0.6610    0.6597      6625



In [39]:
# Both enhancements combined
Xtr, Xte, vec = build_bow(X_train_text, X_test_text, remove_stop=True, lemmatize=True)
print(f'Vocab size (stopwords + lemma): {len(vec.vocabulary_):,}')
results['enhanced_both'] = evaluate('Enhanced: stopwords + lemmatized', Xtr, Xte, y_train, y_test)

Vocab size (stopwords + lemma): 48,422

=== Enhanced: stopwords + lemmatized ===
Test accuracy: 0.6477
              precision    recall  f1-score   support

          -1     0.6137    0.5011    0.5518      1319
           0     0.6248    0.6993    0.6600      2877
           1     0.6950    0.6661    0.6803      2429

    accuracy                         0.6477      6625
   macro avg     0.6445    0.6222    0.6307      6625
weighted avg     0.6484    0.6477    0.6459      6625



## 6. Embeddings System

Each document is represented as the mean of its 300-dim spaCy token vectors and fed to the same classifier.

In [40]:
# Assisted by Claude.

nlp = spacy.load('en_core_web_md', disable=['tagger', 'parser', 'ner', 'lemmatizer', 'attribute_ruler'])
print(f'spaCy model loaded - vector dim: {nlp.vocab.vectors_length}')

def docs_to_vectors(texts, batch_size=256):
    vecs = np.empty((len(texts), nlp.vocab.vectors_length), dtype=np.float32)
    for i, doc in enumerate(nlp.pipe(texts, batch_size=batch_size)):
        vecs[i] = doc.vector
    return vecs

X_tr_emb = docs_to_vectors(X_train_text)
X_te_emb = docs_to_vectors(X_test_text)
print(f'Train embeddings shape: {X_tr_emb.shape}   Test: {X_te_emb.shape}')
results['embeddings'] = evaluate('Embeddings (spaCy mean vector)', X_tr_emb, X_te_emb, y_train, y_test)

spaCy model loaded - vector dim: 300
Train embeddings shape: (26499, 300)   Test: (6625, 300)

=== Embeddings (spaCy mean vector) ===
Test accuracy: 0.5832
              precision    recall  f1-score   support

          -1     0.5690    0.3874    0.4610      1319
           0     0.5719    0.6608    0.6131      2877
           1     0.6042    0.5978    0.6010      2429

    accuracy                         0.5832      6625
   macro avg     0.5817    0.5486    0.5584      6625
weighted avg     0.5832    0.5832    0.5784      6625



## 7. Results summary

In [41]:
summary = pd.DataFrame({
    'system': list(results.keys()),
    'test_accuracy': list(results.values()),
})
summary['delta_vs_baseline'] = summary['test_accuracy'] - results['baseline']
summary = summary.sort_values('test_accuracy', ascending=False).reset_index(drop=True)
summary

,system,test_accuracy,delta_vs_baseline
0,enhanced_lemma,0.660981,0.003774
1,baseline,0.657208,0.000000
2,enhanced_both,0.647698,-0.009509
3,enhanced_stopwords,0.646491,-0.010717
4,embeddings,0.583245,-0.073962


Only lemmatization alone beats the baseline (+0.36 pp). Stopword removal hurts on its own and also drags down the combined system, because stripping `not`/`no`/`n't` removes the main negation cues a unigram BOW relies on. The spaCy mean-vector embeddings perform worst, since averaging dilutes polarity-bearing tokens and "good"/"bad" sit close in vector space.